In [26]:
import pandas as pd
import os
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import numpy as np
import semopy
import json
from scipy import stats
from tqdm import tqdm

In [27]:
path_formulas = os.path.join("..", "utils", "formulas_DAGS_5_conexos.json")
path_modelos_controle = os.path.join("..", "dados", "resultados", "resultados_empiricos", "modelos_controle.csv")
path_modelos_autismo = os.path.join("..", "dados", "resultados", "resultados_empiricos", "modelos_autismo.csv")


path_controle = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_controle_prep.csv")
path_autismo = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_autismo_prep.csv")

In [28]:
colunas_interesse = ["Modelo", "chi2 p-value", "RMSEA", "AIC", "BIC"]
df_modelos_controle = pd.read_csv(path_modelos_controle, index_col="Modelo", usecols=colunas_interesse)
df_modelos_autismo = pd.read_csv(path_modelos_autismo, index_col="Modelo", usecols=colunas_interesse)

df_controle = pd.read_csv(path_controle)
df_autismo = pd.read_csv(path_autismo)

with open(path_formulas, "r") as file:
    formula_modelos_conexos = json.load(file)

### 1) Filtragem de modelos com base em RMSEA < 0.05

In [29]:
df_controle_filtro1 = df_modelos_controle[(df_modelos_controle["RMSEA"] <= 0.05)]
df_autismo_filtro1 = df_modelos_autismo[(df_modelos_autismo["RMSEA"] <= 0.05)]

print(df_controle_filtro1.shape)
print(df_autismo_filtro1.shape)

(217, 4)
(199, 4)


In [30]:
df_controle_filtro1.sort_values("BIC").head(6)

,chi2 p-value,RMSEA,AIC,BIC
Modelo,,,,
M25888_Edges4,1.000000,0.000000,17.999942,56.438838
M00934_Edges4,1.000000,0.000000,17.999978,56.438874
M03528_Edges4,1.000000,0.000000,18.000000,56.438895
M10302_Edges4,1.000000,0.000000,18.000000,56.438896
M00001_Edges4,1.000000,0.000000,18.000000,56.438896
M25373_Edges5,0.223955,0.027218,19.973702,62.683587


In [31]:
df_autismo_filtro1.sort_values("BIC")

,chi2 p-value,RMSEA,AIC,BIC
Modelo,,,,
M10302_Edges4,1.000000,0.0,17.999999,50.872402
M03528_Edges4,1.000000,0.0,18.000000,50.872402
M00001_Edges4,1.000000,0.0,18.000000,50.872402
M00934_Edges4,1.000000,0.0,18.000000,50.872403
M25888_Edges4,1.000000,0.0,18.000000,50.872403
...,...,...,...,...
M26364_Edges9,0.997842,0.0,28.000000,79.134848
M13970_Edges9,0.997869,0.0,28.000000,79.134848
M26185_Edges9,0.997902,0.0,28.000000,79.134848


In [32]:
formula_modelos_conexos["M08812_Edges4"]

{'formula': 'Visual ~ Default_mode + Cerebellar\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Somatomotor + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
 'Arestas': ['Somatomotor->Default_mode',
  'Default_mode->Visual',
  'Cerebellar->Visual',
  'Control->Default_mode'],
 'Num_Arestas': 4}

### 2) Selecionando modelos que passaram na etapa 1, simultaneamente para ambos os conjuntos

In [33]:
modelos_filtro2 = pd.merge(df_controle_filtro1, 
                           df_autismo_filtro1, 
                           how="inner", 
                           left_index=True, 
                           right_index=True, 
                           suffixes=["_controle", "_autismo"])
modelos_filtro2.shape

(136, 8)

### 3) Verificar os TOP 10 modelos com base em ordenação de AIC/BIC

#### 3.1) Ordenação com base no AIC/BIC produzido pelo Conjunto Controle:

In [34]:
modelos_filtro2.sort_values("BIC_controle").head(10)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M25888_Edges4,1.000000,0.000000,17.999942,56.438838,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999978,56.438874,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872402
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872402
M03533_Edges5,0.272771,0.022695,19.975956,62.685840,0.171775,0.043844,19.945757,56.470648
M00948_Edges5,0.272774,0.022695,19.975956,62.685840,0.171776,0.043844,19.945757,56.470648
M25663_Edges6,0.193071,0.031401,21.977004,68.957877,0.323962,0.024109,21.967296,62.144677
M25251_Edges6,0.193072,0.031401,21.977004,68.957877,0.323958,0.024110,21.967296,62.144677


Percebe-se que o Top 5 tem uma discrepância em BIC relevante a partir do 6º modelo

In [35]:
modelos_filtro2.sort_values("BIC_autismo").head(6)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872402
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872402
M00934_Edges4,1.000000,0.000000,17.999978,56.438874,1.000000,0.000000,18.000000,50.872403
M25888_Edges4,1.000000,0.000000,17.999942,56.438838,1.000000,0.000000,18.000000,50.872403
M03533_Edges5,0.272771,0.022695,19.975956,62.685840,0.171775,0.043844,19.945757,56.470648


In [36]:
modelos_filtro2.sort_values("AIC_controle").head(6)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M25888_Edges4,1.000000,0.000000,17.999942,56.438838,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,17.999978,56.438874,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872402
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872402
M03533_Edges5,0.272771,0.022695,19.975956,62.685840,0.171775,0.043844,19.945757,56.470648


#### 3.2) Ordenação com base no AIC/BIC produzido pelo Conjunto Autismo:

In [37]:
modelos_filtro2.sort_values("BIC_autismo").head(6)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872402
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872402
M00934_Edges4,1.000000,0.000000,17.999978,56.438874,1.000000,0.000000,18.000000,50.872403
M25888_Edges4,1.000000,0.000000,17.999942,56.438838,1.000000,0.000000,18.000000,50.872403
M03533_Edges5,0.272771,0.022695,19.975956,62.685840,0.171775,0.043844,19.945757,56.470648


Percebe-se que o Top 5 tem uma discrepância em BIC relevante a partir do 6º modelo

In [22]:
display(modelos_filtro2.sort_values("BIC_autismo").head(6))
top_5 = modelos_filtro2.sort_values("BIC_autismo").head(5)

,chi2 p-value_controle,CFI_controle,GFI_controle,AGFI_controle,NFI_controle,TLI_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,CFI_autismo,GFI_autismo,AGFI_autismo,NFI_autismo,TLI_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,,,,,,,,,,,
M10302_Edges4,1.000000,1.010848,1.000000,1.000000,1.000000,1.018081,0.000000,18.000000,56.438896,1.000000,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,17.999999,50.872402
M03528_Edges4,1.000000,1.010848,1.000000,1.000000,1.000000,1.018081,0.000000,18.000000,56.438895,1.000000,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,18.000000,50.872402
M00001_Edges4,1.000000,1.010849,1.000000,1.000000,1.000000,1.018081,0.000000,18.000000,56.438896,1.000000,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,18.000000,50.872402
M00934_Edges4,1.000000,1.010838,0.999990,0.999983,0.999990,1.018064,0.000000,17.999978,56.438874,1.000000,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,18.000000,50.872403
M25888_Edges4,1.000000,1.010821,0.999973,0.999955,0.999973,1.018035,0.000000,17.999942,56.438838,1.000000,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,18.000000,50.872403
M03533_Edges5,0.272771,0.997541,0.988705,0.977410,0.988705,0.995083,0.022695,19.975956,62.685840,0.171775,0.994814,0.985588,0.971176,0.985588,0.989628,0.043844,19.945757,56.470648


### 5) Verificação de arestas estatisticamente significativas

In [23]:
def roda_os_modelos(df, modelos, ajuste_p=None, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        try:
            formula = modelo["formula"]
        except:
            formula = modelo
        
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            
            if ajuste_p:
                edges_inspection = edges_inspection[edges_inspection["op"] == "~"]
                p_valores = edges_inspection["p-value"]
                rejeitados, p_corrigidos_fdr, _, _ = multipletests(p_valores, alpha=0.05, method=ajuste_p)
                edges_inspection["p-value-adj"] = p_corrigidos_fdr
                edges_inspection["tipo_correcao"] = ajuste_p
                edges_inspection["rejeita"] = rejeitados
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados

In [24]:
modelos_interesse = {m:v for m,v in formula_modelos_conexos.items() if m in top_5.index}

#### 4.1) Arestas indicadas pelo Conjunto de Controle

In [25]:
_, dic_arestas_controle = roda_os_modelos(df_controle, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_controle[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.077541,0.026430,2.933806,3.348332e-03,1.339333e-02,bonferroni,True
1,Control,~,Somatomotor,0.112365,0.020952,5.363050,8.182845e-08,3.273138e-07,bonferroni,True
2,Control,~,Default_mode,0.198148,0.018425,10.754478,0.000000e+00,0.000000e+00,bonferroni,True
3,Control,~,Cerebellar,0.137718,0.026242,5.247908,1.538365e-07,6.153458e-07,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.255639,0.041580,6.148104,7.841450e-10,3.136580e-09,bonferroni,True
1,Cerebellar,~,Somatomotor,0.209462,0.033537,6.245725,4.218386e-10,1.687354e-09,bonferroni,True
2,Cerebellar,~,Default_mode,-0.093238,0.032604,-2.859705,4.240349e-03,1.696140e-02,bonferroni,True
3,Cerebellar,~,Control,0.359354,0.068473,5.248091,1.536835e-07,6.147339e-07,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.308735,0.055429,5.569905,2.548785e-08,1.019514e-07,bonferroni,True
1,Default_mode,~,Somatomotor,0.256296,0.044677,5.736669,9.655659e-09,3.862264e-08,bonferroni,True
2,Default_mode,~,Cerebellar,-0.163690,0.057176,-2.862917,4.197605e-03,1.679042e-02,bonferroni,True
3,Default_mode,~,Control,0.905912,0.084306,10.745480,0.000000e+00,0.000000e+00,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.205308,0.052443,3.914842,9.046346e-05,3.618539e-04,bonferroni,True
1,Somatomotor,~,Default_mode,0.229243,0.039384,5.820686,5.860648e-09,2.344259e-08,bonferroni,True
2,Somatomotor,~,Cerebellar,0.327913,0.051844,6.324941,2.533285e-10,1.013314e-09,bonferroni,True
3,Somatomotor,~,Control,0.458974,0.084512,5.430878,5.607743e-08,2.243097e-07,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.133829,0.034630,3.864483,1.113248e-04,4.452991e-04,bonferroni,True
1,Visual,~,Default_mode,0.179758,0.032266,5.571144,2.530720e-08,1.012288e-07,bonferroni,True
2,Visual,~,Cerebellar,0.260815,0.042446,6.144675,8.012739e-10,3.205096e-09,bonferroni,True
3,Visual,~,Control,0.206476,0.070399,2.932950,3.357584e-03,1.343034e-02,bonferroni,True


#### 4.2) Arestas indicadas pelo Conjunto de Autismo

In [26]:
_, dic_arestas_autismo = roda_os_modelos(df_autismo, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_autismo[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.069191,0.034291,2.017792,0.043613,0.174452,bonferroni,False
1,Control,~,Somatomotor,0.116306,0.029889,3.891295,0.000100,0.000399,bonferroni,True
2,Control,~,Default_mode,0.224086,0.025521,8.780503,0.000000,0.000000,bonferroni,True
3,Control,~,Cerebellar,0.154121,0.034000,4.533023,0.000006,0.000023,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.232054,0.056460,4.110090,3.955056e-05,1.582023e-04,bonferroni,True
1,Cerebellar,~,Somatomotor,0.294320,0.048574,6.059234,1.367715e-09,5.470858e-09,bonferroni,True
2,Cerebellar,~,Default_mode,-0.173507,0.047299,-3.668288,2.441797e-04,9.767186e-04,bonferroni,True
3,Cerebellar,~,Control,0.436381,0.096268,4.532970,5.816018e-06,2.326407e-05,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.378520,0.067487,5.608754,2.037889e-08,8.151558e-08,bonferroni,True
1,Default_mode,~,Somatomotor,0.367429,0.059290,6.197148,5.749552e-10,2.299821e-09,bonferroni,True
2,Default_mode,~,Cerebellar,-0.259661,0.070839,-3.665495,2.468605e-04,9.874421e-04,bonferroni,True
3,Default_mode,~,Control,0.950226,0.108224,8.780156,0.000000e+00,0.000000e+00,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.060357,0.066618,0.906029,3.649207e-01,1.000000e+00,bonferroni,False
1,Somatomotor,~,Default_mode,0.323179,0.052166,6.195205,5.820924e-10,2.328369e-09,bonferroni,True
2,Somatomotor,~,Cerebellar,0.387736,0.064007,6.057692,1.380883e-09,5.523534e-09,bonferroni,True
3,Somatomotor,~,Control,0.433799,0.111517,3.889979,1.002528e-04,4.010112e-04,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.047730,0.052049,0.917020,3.591324e-01,1.000000e+00,bonferroni,False
1,Visual,~,Default_mode,0.262686,0.046392,5.662250,1.494009e-08,5.976035e-08,bonferroni,True
2,Visual,~,Cerebellar,0.241391,0.058128,4.152761,3.284881e-05,1.313952e-04,bonferroni,True
3,Visual,~,Control,0.203513,0.099961,2.035924,4.175795e-02,1.670318e-01,bonferroni,False


### 6) Poda de modelos e comparação com modelos não podados

#### 6.1) Poda de arestas sem Bonferroni

In [17]:
modelos_podados_bonf = {
    "M00934_Edges4_poda_CeDM": 'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control\nControl ~~ Control',
    "M03528_Edges4_poda_DMCe":'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M00001_Edges4_poda_CV": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~ 0*Visual + Somatomotor + Default_mode + Cerebellar",
    "M10302_Edges4_poda_SV": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_poda_VC": 'Visual ~ Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M25888_Edges4_poda_VS_VC": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control"
}

modelos_podados_sem_bonf = {
    "M03528_Edges4_poda_DMCe":'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M00934_Edges4_poda_CeDM": 'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control\nControl ~~ Control',
    "M10302_Edges4_poda_SV": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_poda_VS": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
}

In [ ]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo, 
                                        modelos_podados_sem_bonf,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle, 
                                        modelos_podados_sem_bonf,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados_sem_bonf.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M03528_Edges4_poda_DMCe,0.872020,0.000000,15.988139,50.156046,0.281594,0.028491,15.939553,45.159467
M00934_Edges4_poda_CeDM,0.872020,0.000000,15.988139,50.156046,0.281595,0.028491,15.939553,45.159467
M10302_Edges4_poda_SV,0.146100,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572
M25888_Edges4_poda_VS,0.146098,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572



M03528_Edges4_poda_DMCe:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Default_mode,~,Visual,0.168610,0.040301,4.183804,0.000029
1,Default_mode,~,Somatomotor,0.183836,0.041912,4.386243,0.000012
2,Default_mode,~,Cerebellar,0.000000,-,-,-
3,Default_mode,~,Control,0.343209,0.041729,8.224666,0.0
4,Visual,~~,Visual,1.000000,0.047144,21.211512,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.044486,22.478791,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.047441,21.07883,0.0
7,Control,~~,Control,1.000000,0.045835,21.81727,0.0
8,Default_mode,~~,Default_mode,0.699289,0.042998,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Default_mode,~,Visual,0.200356,0.049395,4.056216,0.00005
1,Default_mode,~,Somatomotor,0.247415,0.053159,4.654283,0.000003
2,Default_mode,~,Cerebellar,0.000000,-,-,-
3,Default_mode,~,Control,0.402979,0.054294,7.422185,0.0
4,Visual,~~,Visual,1.000000,0.057694,17.33276,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.047359,21.11546,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.058293,17.154709,0.0
7,Control,~~,Control,1.000000,0.048736,20.518568,0.0
8,Default_mode,~~,Default_mode,0.491248,0.041152,11.937336,0.0


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Somatomotor,~,Visual,0.000000,-,-,-
1,Somatomotor,~,Default_mode,0.338159,0.055529,6.089789,0.0
2,Somatomotor,~,Cerebellar,0.304860,0.04703,6.482203,0.0
3,Somatomotor,~,Control,0.239095,0.059334,4.029638,0.000056
4,Visual,~~,Visual,1.000000,0.054296,18.417433,0.0
5,Default_mode,~~,Default_mode,1.000000,0.045986,21.745947,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.063682,15.702924,0.0
7,Control,~~,Control,1.000000,0.043198,23.14942,0.0
8,Somatomotor,~~,Somatomotor,0.507824,0.042541,11.937336,0.0



M25888_Edges4_poda_VS:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.231637,0.043143,5.368992,0.0
2,Visual,~,Cerebellar,0.287832,0.04013,7.172468,0.0
3,Visual,~,Control,0.150542,0.045019,3.343946,0.000826
4,Somatomotor,~~,Somatomotor,1.000000,0.043569,22.951885,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044877,22.283314,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.050487,19.807011,0.0
7,Control,~~,Control,1.000000,0.041151,24.300592,0.0
8,Visual,~~,Visual,0.751627,0.046216,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.334177,0.062547,5.342852,0.0
2,Visual,~,Cerebellar,0.229869,0.052974,4.339286,0.000014
3,Visual,~,Control,0.180521,0.066833,2.701085,0.006911
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.644291,0.053973,11.937336,0.0


#### 6.2) Poda com Bonferroni

In [19]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo_prep, 
                                        modelos_podados_bonf,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle_prep, 
                                        modelos_podados_bonf,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados_bonf.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M00934_Edges4_poda_CeDM,0.872020,0.000000,15.988139,50.156046,0.281595,0.028491,15.939553,45.159467
M03528_Edges4_poda_DMCe,0.872020,0.000000,15.988139,50.156046,0.281594,0.028491,15.939553,45.159467
M00001_Edges4_poda_CV,0.519881,0.000000,15.976668,50.144575,0.625919,0.000000,15.962952,45.182866
M10302_Edges4_poda_SV,0.146100,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572
M25888_Edges4_poda_VC,0.519872,0.000000,15.976667,50.144575,0.625918,0.000000,15.962952,45.182866
M25888_Edges4_poda_VS_VC,0.005147,0.057319,13.917286,43.814205,0.320170,0.023627,13.934959,39.502384



M00934_Edges4_poda_CeDM:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.246933,0.042269,5.841895,0.0
1,Cerebellar,~,Somatomotor,0.217937,0.043959,4.957703,0.000001
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.155841,0.043768,3.560657,0.00037
4,Visual,~~,Visual,1.000000,0.04852,20.610244,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.04482,22.311603,0.0
6,Default_mode,~~,Default_mode,1.000000,0.043312,23.08834,0.0
7,Control,~~,Control,1.000000,0.041757,23.948244,0.0
8,Cerebellar,~~,Cerebellar,0.769273,0.047301,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.170136,0.058662,2.900285,0.003728
1,Cerebellar,~,Somatomotor,0.332789,0.063132,5.271331,0.0
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.158600,0.06448,2.459675,0.013906
4,Visual,~~,Visual,1.000000,0.056071,17.834636,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.047788,20.925872,0.0
6,Default_mode,~~,Default_mode,1.000000,0.04182,23.912237,0.0
7,Control,~~,Control,1.000000,0.041761,23.945791,0.0
8,Cerebellar,~~,Cerebellar,0.692870,0.058042,11.937336,0.0


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.130932,0.065261,2.006299,0.044824
1,Visual,~,Default_mode,0.375060,0.058689,6.390609,0.0
2,Visual,~,Cerebellar,0.231870,0.055445,4.18199,0.000029
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.651603,0.054585,11.937336,0.0



M25888_Edges4_poda_VS_VC:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.297029,0.038833,7.648893,0.0
2,Visual,~,Cerebellar,0.326398,0.038833,8.405191,0.0
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043569,22.951885,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044877,22.283314,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.050487,19.807011,0.0
7,Control,~~,Control,1.000000,0.041151,24.300592,0.0
8,Visual,~~,Visual,0.767336,0.047182,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.437084,0.050221,8.703171,0.0
2,Visual,~,Cerebellar,0.280427,0.050221,5.583835,0.0
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.661166,0.055386,11.937336,0.0


### 7) Teste de diferença de parâmetros das arestas:

#### 7.1) Teste z

In [27]:
for nome, formula in modelos_interesse.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula["formula"])
    model_controle.fit(df_controle)
    arestas_controle = model_controle.inspect()
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula["formula"])
    model_autismo.fit(df_autismo)
    arestas_autismo = model_autismo.inspect()
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()

    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # Cálculo do Z e P-valor usando as colunas mescladas
    z_diff = (df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]) / np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2))
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor"] = p_valor
    
    print(nome)
    colunas = ["lval", "rval", "Estimate_ctrl", "p-value_ctrl", "Estimate_tea", "p-value_tea", "p_valor"]
    display(df_comparacao[colunas])

M00001_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Control,Visual,0.077541,3.348332e-03,0.069191,0.043613,0.847077
1,Control,Somatomotor,0.112365,8.182845e-08,0.116306,0.000100,0.914023
2,Control,Default_mode,0.198148,0.000000e+00,0.224086,0.000000,0.409910
3,Control,Cerebellar,0.137718,1.538365e-07,0.154121,0.000006,0.702509


M00934_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Cerebellar,Visual,0.255639,7.841450e-10,0.232054,3.955056e-05,0.736604
1,Cerebellar,Somatomotor,0.209462,4.218386e-10,0.294320,1.367715e-09,0.150539
2,Cerebellar,Default_mode,-0.093238,4.240349e-03,-0.173507,2.441797e-04,0.162340
3,Cerebellar,Control,0.359354,1.536835e-07,0.436381,5.816018e-06,0.514393


M03528_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Default_mode,Visual,0.308735,2.548785e-08,0.378520,2.037889e-08,0.424251
1,Default_mode,Somatomotor,0.256296,9.655659e-09,0.367429,5.749552e-10,0.134400
2,Default_mode,Cerebellar,-0.163690,4.197605e-03,-0.259661,2.468605e-04,0.291783
3,Default_mode,Control,0.905912,0.000000e+00,0.950226,0.000000e+00,0.746678


M10302_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Somatomotor,Visual,0.205308,9.046346e-05,0.060357,3.649207e-01,0.087330
1,Somatomotor,Default_mode,0.229243,5.860648e-09,0.323179,5.820924e-10,0.150681
2,Somatomotor,Cerebellar,0.327913,2.533285e-10,0.387736,1.380883e-09,0.467672
3,Somatomotor,Control,0.458974,5.607743e-08,0.433799,1.002528e-04,0.857214


M25888_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Visual,Somatomotor,0.133829,1.113248e-04,0.047730,3.591324e-01,0.168443
1,Visual,Default_mode,0.179758,2.530720e-08,0.262686,1.494009e-08,0.142243
2,Visual,Cerebellar,0.260815,8.012739e-10,0.241391,3.284881e-05,0.787265
3,Visual,Control,0.206476,3.357584e-03,0.203513,4.175795e-02,0.980665


In [28]:
# Z-score para 95% de confiança
Z_95 = 1.96 

for nome, formula in modelos_interesse.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula["formula"])
    model_controle.fit(df_controle)
    arestas_controle = model_controle.inspect()
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()

    # Garantir que Std. Err é numérico (previne erros em algumas versões do semopy)
    arestas_controle["Std. Err"] = pd.to_numeric(arestas_controle["Std. Err"])

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula["formula"])
    model_autismo.fit(df_autismo)
    arestas_autismo = model_autismo.inspect()
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()
    
    # Garantir que Std. Err é numérico
    arestas_autismo["Std. Err"] = pd.to_numeric(arestas_autismo["Std. Err"])

    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # =====================================================================
    # NOVAS ADIÇÕES: CÁLCULO DOS INTERVALOS DE CONFIANÇA (95%)
    # =====================================================================
    
    # IC 95% Grupo Controle
    df_comparacao["IC_inf_ctrl"] = df_comparacao["Estimate_ctrl"] - (Z_95 * df_comparacao["Std. Err_ctrl"])
    df_comparacao["IC_sup_ctrl"] = df_comparacao["Estimate_ctrl"] + (Z_95 * df_comparacao["Std. Err_ctrl"])
    
    # IC 95% Grupo Autismo (TEA)
    df_comparacao["IC_inf_tea"] = df_comparacao["Estimate_tea"] - (Z_95 * df_comparacao["Std. Err_tea"])
    df_comparacao["IC_sup_tea"] = df_comparacao["Estimate_tea"] + (Z_95 * df_comparacao["Std. Err_tea"])
    
    # SE da Diferença (já usado no seu cálculo do Z)
    se_diff = np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2)) 
    df_comparacao["diff"] = df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]
    
    # IC 95% da Diferença entre os grupos
    df_comparacao["IC_inf_diff"] = df_comparacao["diff"] - (Z_95 * se_diff)
    df_comparacao["IC_sup_diff"] = df_comparacao["diff"] + (Z_95 * se_diff)
    # =====================================================================

    # Cálculo do Z e P-valor usando as colunas mescladas (Seu código original)
    z_diff = df_comparacao["diff"] / se_diff
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor_diff"] = p_valor # Renomeei levemente para deixar claro que é o p-valor da diferença
    
    print(f"\nResultados para o modelo: {nome}")
    
    # Atualizei a lista de colunas para exibir os novos limites
    colunas = [
        "lval", "rval", 
        "Estimate_ctrl", "IC_inf_ctrl", "IC_sup_ctrl", "p-value_ctrl", 
        "Estimate_tea", "IC_inf_tea", "IC_sup_tea", "p-value_tea", 
        "IC_inf_diff", "IC_sup_diff", "p_valor_diff"
    ]
    
    # Formatação opcional para não exibir muitas casas decimais
    display(df_comparacao[colunas].round(4))
    display(df_comparacao[["lval", "rval", "diff", "IC_inf_diff", "IC_sup_diff", "z_diff", "p_valor_diff"]])


Resultados para o modelo: M00001_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Control,Visual,0.0775,0.0257,0.1293,0.0033,0.0692,0.0020,0.1364,0.0436,-0.0765,0.0932,0.8471
1,Control,Somatomotor,0.1124,0.0713,0.1534,0.0000,0.1163,0.0577,0.1749,0.0001,-0.0755,0.0676,0.9140
2,Control,Default_mode,0.1981,0.1620,0.2343,0.0000,0.2241,0.1741,0.2741,0.0000,-0.0876,0.0358,0.4099
3,Control,Cerebellar,0.1377,0.0863,0.1892,0.0000,0.1541,0.0875,0.2208,0.0000,-0.1006,0.0678,0.7025


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Control,Visual,0.008349,-0.076508,0.093206,0.192850,0.847077
1,Control,Somatomotor,-0.003941,-0.075482,0.067601,-0.107965,0.914023
2,Control,Default_mode,-0.025938,-0.087633,0.035756,-0.824052,0.409910
3,Control,Cerebellar,-0.016404,-0.100585,0.067777,-0.381936,0.702509



Resultados para o modelo: M00934_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Cerebellar,Visual,0.2556,0.1741,0.3371,0.0000,0.2321,0.1214,0.3427,0.0000,-0.1138,0.1610,0.7366
1,Cerebellar,Somatomotor,0.2095,0.1437,0.2752,0.0000,0.2943,0.1991,0.3895,0.0000,-0.2006,0.0308,0.1505
2,Cerebellar,Default_mode,-0.0932,-0.1571,-0.0293,0.0042,-0.1735,-0.2662,-0.0808,0.0002,-0.0323,0.1929,0.1623
3,Cerebellar,Control,0.3594,0.2251,0.4936,0.0000,0.4364,0.2477,0.6251,0.0000,-0.3086,0.1545,0.5144


...

,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Somatomotor,Visual,0.2053,0.1025,0.3081,0.0001,0.0604,-0.0702,0.1909,0.3649,-0.0212,0.3111,0.0873
1,Somatomotor,Default_mode,0.2292,0.1520,0.3064,0.0000,0.3232,0.2209,0.4254,0.0000,-0.2220,0.0342,0.1507
2,Somatomotor,Cerebellar,0.3279,0.2263,0.4295,0.0000,0.3877,0.2623,0.5132,0.0000,-0.2213,0.1016,0.4677
3,Somatomotor,Control,0.4590,0.2933,0.6246,0.0000,0.4338,0.2152,0.6524,0.0001,-0.2491,0.2994,0.8572


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Somatomotor,Visual,0.144950,-0.021225,0.311126,1.709656,0.087330
1,Somatomotor,Default_mode,-0.093936,-0.222048,0.034177,-1.437129,0.150681
2,Somatomotor,Cerebellar,-0.059823,-0.221267,0.101622,-0.726272,0.467672
3,Somatomotor,Control,0.025175,-0.249073,0.299423,0.179921,0.857214



Resultados para o modelo: M25888_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Visual,Somatomotor,0.1338,0.0660,0.2017,0.0001,0.0477,-0.0543,0.1497,0.3591,-0.0364,0.2086,0.1684
1,Visual,Default_mode,0.1798,0.1165,0.2430,0.0000,0.2627,0.1718,0.3536,0.0000,-0.1937,0.0278,0.1422
2,Visual,Cerebellar,0.2608,0.1776,0.3440,0.0000,0.2414,0.1275,0.3553,0.0000,-0.1216,0.1605,0.7873
3,Visual,Control,0.2065,0.0685,0.3445,0.0034,0.2035,0.0076,0.3994,0.0418,-0.2367,0.2426,0.9807


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Visual,Somatomotor,0.086099,-0.036433,0.208632,1.377224,0.168443
1,Visual,Default_mode,-0.082927,-0.193687,0.027832,-1.467489,0.142243
2,Visual,Cerebellar,0.019424,-0.121649,0.160496,0.269864,0.787265
3,Visual,Control,0.002963,-0.236672,0.242598,0.024235,0.980665


### 8) Teste de Invariância global (Teste de Chow)

Utilizando dados sem Pré-processamento

In [29]:
path_controle = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_controle.csv")
path_autismo = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_autismo.csv")

df_controle = pd.read_csv(path_controle)
df_autismo = pd.read_csv(path_autismo)

In [30]:
# Faz sentido retirar o intercepto?
modelos_estrela = {
    "M00001_Edges4": "Control ~ Visual + Somatomotor + Default_mode + Cerebellar",
    "M00934_Edges4": "Cerebellar ~ Visual + Somatomotor + Default_mode + Control",
    "M03528_Edges4": "Default_mode ~ Visual + Somatomotor + Cerebellar + Control",
    "M10302_Edges4": "Somatomotor ~ Visual + Default_mode + Cerebellar + Control",
    "M25888_Edges4": "Visual ~ Somatomotor + Default_mode + Cerebellar + Control",
    }


df_total = pd.concat([df_controle, df_autismo])
retira_va_indep = ""

for nome, formula in modelos_estrela.items():

    # 1. Ajustar os modelos via OLS (Mínimos Quadrados Ordinários)
    modelo_total = smf.ols(formula, data=df_total, drop_cols=retira_va_indep).fit()
    modelo_controle = smf.ols(formula, data=df_controle, drop_cols=retira_va_indep).fit()
    modelo_autismo = smf.ols(formula, data=df_autismo, drop_cols=retira_va_indep).fit()

    # 2. Extrair a Soma dos Quadrados dos Resíduos (SSR - a métrica de erro)
    ssr_total = modelo_total.ssr
    ssr_controle = modelo_controle.ssr
    ssr_autismo = modelo_autismo.ssr


    # 3. Definir os parâmetros para o teste
    k = 5 # 4 preditores + 1 intercepto
    N_controle = len(df_controle) # 529
    N_autismo = len(df_autismo)   # 285

    # 4. Calcular a Estatística F:
    numerador = (ssr_total - (ssr_controle + ssr_autismo)) / k
    denominador = (ssr_controle + ssr_autismo) / (N_controle + N_autismo - 2 * k)
    estatistica_f = numerador / denominador

    # 5. Calcular o P-valor Global:
    graus_liberdade_numerador = k # 5 variáveis
    graus_liberdade_denominador = N_controle + N_autismo - 2 * k
    p_valor = stats.f.sf(estatistica_f, graus_liberdade_numerador, graus_liberdade_denominador)

    print(f"\n\nTESTANDO O MODELO {nome}")
    print(f"\nEstatística F: {estatistica_f:.4f}")
    print(f"P-valor Global: {p_valor:.6e}")
    print("-" * 55)

    if p_valor < 0.05:
        print("VEREDITO: Quebra Estrutural Detectada (Diferença Sistêmica).")
        print("A topologia de predição da Default_mode difere estatisticamente entre Controle e Autismo.")
    else:
        print("VEREDITO: Invariância Confirmada.")
        print("Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.")



TESTANDO O MODELO M00001_Edges4

Estatística F: 0.8034
P-valor Global: 5.473196e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M00934_Edges4

Estatística F: 1.9034
P-valor Global: 9.142191e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M03528_Edges4

Estatística F: 1.9862
P-valor Global: 7.844857e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M10302_Edges4

Estatística F: 0.8720
P-valor Global: 4.993383e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-r

### 9) Regressão linear - teste de parâmetros

In [23]:
resultados_controle = {}
resultados_autismo = {}


for nome_modelo, formula in modelos_estrela.items():
    print(f"===========================================================")
    print(f"Avaliando Modelo: {nome_modelo}")
    print(f"Fórmula: {formula}")
    print(f"===========================================================")
    
    # smf.ols = Ordinary Least Squares (Regressão Linear Múltipla)
    # .fit() ajusta os dados aos parâmetros da regressão
    modelo_controle = smf.ols(formula=formula, data=df_controle).fit()
    modelo_autismo = smf.ols(formula=formula, data=df_autismo).fit()

    
    # Salvando o modelo na nossa lista de resultados para consultas futuras
    resultados_controle[nome_modelo] = modelo_controle
    resultados_autismo[nome_modelo] = modelo_autismo

    print("\nCONTROLE")
    print(modelo_controle.summary())
    print("\n")

    print("AUTISMO")
    print(modelo_autismo.summary())
    print("\n")



Avaliando Modelo: M00001_Edges4
Fórmula: Control ~ Visual + Somatomotor + Default_mode + Cerebellar

CONTROLE
                            OLS Regression Results                            
Dep. Variable:                Control   R-squared:                       0.542
Model:                            OLS   Adj. R-squared:                  0.538
Method:                 Least Squares   F-statistic:                     154.9
Date:                Wed, 29 Apr 2026   Prob (F-statistic):           2.26e-87
Time:                        07:48:13   Log-Likelihood:                -1382.5
No. Observations:                 529   AIC:                             2775.
Df Residuals:                     524   BIC:                             2796.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

In [24]:
resultados_controle = {}
resultados_autismo = {}
resultados_diferenca = {} # Dicionário para salvar os testes de diferença

for nome_modelo, formula in modelos_estrela.items():
    print(f"===========================================================")
    print(f"Avaliando Modelo: {nome_modelo}")
    print(f"Fórmula: {formula}")
    print(f"===========================================================")
    
    # Ajustando os modelos
    modelo_controle = smf.ols(formula=formula, data=df_controle).fit()
    modelo_autismo = smf.ols(formula=formula, data=df_autismo).fit()
    
    # Salvando os resultados
    resultados_controle[nome_modelo] = modelo_controle
    resultados_autismo[nome_modelo] = modelo_autismo

    # ---------------------------------------------------------
    # TESTE DE DIFERENÇA DOS PARÂMETROS
    # ---------------------------------------------------------
    print("\n--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---")
    
    diferencas_modelo = []
    
    # Iterar por cada variável preditora usada na fórmula (incluindo o Intercept)
    for parametro in modelo_controle.params.index:
        # Extraindo os Betas (Coeficientes)
        beta_c = modelo_controle.params[parametro]
        beta_a = modelo_autismo.params[parametro]
        
        # Extraindo os Erros Padrão (Standard Errors - bse)
        se_c = modelo_controle.bse[parametro]
        se_a = modelo_autismo.bse[parametro]
        
        # Calculando a Estatística Z
        z_stat = (beta_c - beta_a) / np.sqrt(se_c**2 + se_a**2)
        
        # Calculando o P-valor (Bicaudal)
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        # Marcador visual de significância
        sig = "*" if p_value < 0.05 else ""
        
        print(f"Parâmetro: {parametro:15} | Z-Score: {z_stat:>8.4f} | P-valor: {p_value:>8.4f} {sig}")
        
        # Guardando em uma lista caso queira transformar em DataFrame depois
        diferencas_modelo.append({
            'Variavel': parametro,
            'Beta_Controle': beta_c,
            'Beta_Autismo': beta_a,
            'Z_Stat': z_stat,
            'P_Value': p_value
        })
        
    resultados_diferenca[nome_modelo] = pd.DataFrame(diferencas_modelo)
    print("----------------------------------------------------------------\n")

Avaliando Modelo: M00001_Edges4
Fórmula: Control ~ Visual + Somatomotor + Default_mode + Cerebellar

--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---
Parâmetro: Intercept       | Z-Score:   1.6893 | P-valor:   0.0912 
Parâmetro: Visual          | Z-Score:   0.1034 | P-valor:   0.9176 
Parâmetro: Somatomotor     | Z-Score:  -0.1525 | P-valor:   0.8788 
Parâmetro: Default_mode    | Z-Score:  -0.8535 | P-valor:   0.3934 
Parâmetro: Cerebellar      | Z-Score:  -0.3992 | P-valor:   0.6897 
----------------------------------------------------------------

Avaliando Modelo: M00934_Edges4
Fórmula: Cerebellar ~ Visual + Somatomotor + Default_mode + Control

--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---
Parâmetro: Intercept       | Z-Score:   0.7571 | P-valor:   0.4490 
Parâmetro: Visual          | Z-Score:   0.2970 | P-valor:   0.7665 
Parâmetro: Somatomotor     | Z-Score:  -1.4515 | P-valor:   0.1466 
Parâmetro: Default_mode    | Z-Score:   1.4866 | P-valor: 

### 10) Bootstrap

In [57]:
modelo_ganhador = {"formula_ols":"Visual ~ Somatomotor + Default_mode + Cerebellar + Control",
                   "formula_sem": "Visual ~ Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
                   "arestas":["Somatomotor->Visual", "Default_mode->Visual", "Cerebellar->Visual", "Control->Visual"]}

#### 10.1) Contrução de Intervalos de Confiança

In [58]:
def bootstrap_estimativas(modelo: dict, dados: pd.DataFrame, n_sims: int) -> dict:
    formula = modelo["formula_sem"]
    estimativas_arestas = {a:[] for a in modelo["arestas"]}
    model = semopy.Model(formula)
    for i in tqdm(range(n_sims)):
        reamostragem = dados.sample(n=len(dados), replace=True)
        model.fit(reamostragem)
        parametros = model.inspect()
        for aresta in estimativas_arestas.keys():
            split = aresta.split("->")
            lval = split[1]
            rval = split[0]
            dados_aresta = parametros[(parametros["rval"] == rval) & (parametros["lval"] == lval)]
            estimativas_arestas[aresta].append(dados_aresta["Estimate"].values[0].item())

    return estimativas_arestas

In [79]:
estimativas_controle = bootstrap_estimativas(modelo_ganhador, df_controle_prep, n_sims=1000)
estimatvas_autismo = bootstrap_estimativas(modelo_ganhador, df_autismo_prep, n_sims=1000)

100%|██████████| 1000/1000 [00:09<00:00, 101.40it/s]


In [80]:
print("CONTROLE")
for nome_aresta, lista in estimativas_controle.items():
    ic = np.percentile(lista, [2.5, 97.5])
    print(f"{nome_aresta}: {np.mean(lista):.4f} [{ic[0].item():.4f}, {ic[1].item():.4f}]")


print("\n\nAUTISMO")
for nome_aresta, lista in estimatvas_autismo.items():
    ic = np.percentile(lista, [2.5, 97.5])
    print(f"{nome_aresta}: {np.mean(lista):.4f} [{ic[0].item():.4f}, {ic[1].item():.4f}]")

CONTROLE
Somatomotor->Visual: 0.1463 [0.0576, 0.2364]
Default_mode->Visual: 0.1981 [0.1112, 0.2810]
Cerebellar->Visual: 0.2506 [0.1717, 0.3291]
Control->Visual: 0.1150 [0.0201, 0.2045]


AUTISMO
Somatomotor->Visual: 0.0936 [-0.0409, 0.2276]
Default_mode->Visual: 0.3010 [0.1771, 0.4204]
Cerebellar->Visual: 0.2013 [0.0861, 0.3217]
Control->Visual: 0.1596 [0.0371, 0.2867]


In [15]:
modelo_ganhador = {"M28888": "Visual ~ Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control"}
res_controle, dic_arestas_controle = roda_os_modelos(df_controle, 
                                                     modelo_ganhador, 
                                                     ajuste_p="bonf", 
                                                     retornar_inspection=True)

res_autismo, dic_arestas_autismo = roda_os_modelos(df_autismo, 
                                                     modelo_ganhador, 
                                                     ajuste_p="bonf", 
                                                     retornar_inspection=True)

In [16]:
arestas_mod_ganhador_controle = dic_arestas_controle["M28888"].copy()
arestas_mod_ganhador_autismo = dic_arestas_autismo["M28888"].copy()

In [17]:
arestas_mod_ganhador_controle["ic_inf"] = arestas_mod_ganhador_controle["Estimate"] - 1.96 * arestas_mod_ganhador_controle["Std. Err"]
arestas_mod_ganhador_controle["ic_sup"] = arestas_mod_ganhador_controle["Estimate"] + 1.96 * arestas_mod_ganhador_controle["Std. Err"]

arestas_mod_ganhador_autismo["ic_inf"] = arestas_mod_ganhador_autismo["Estimate"] - 1.96 * arestas_mod_ganhador_autismo["Std. Err"]
arestas_mod_ganhador_autismo["ic_sup"] = arestas_mod_ganhador_autismo["Estimate"] + 1.96 * arestas_mod_ganhador_autismo["Std. Err"]

ordem_colunas = ["lval", "op", "rval", "Estimate", "ic_inf", "ic_sup", "p-value", "p-value-adj", "rejeita", "tipo_correcao"]
print("CONTROLE - M28888")
display(arestas_mod_ganhador_controle[ordem_colunas])

print("\n\nAUTISMO - M28888")
display(arestas_mod_ganhador_autismo[ordem_colunas])

CONTROLE - M28888


,lval,op,rval,Estimate,ic_inf,ic_sup,p-value,p-value-adj,rejeita,tipo_correcao
0,Visual,~,Somatomotor,0.133829,0.065953,0.201705,1.113248e-04,4.452991e-04,True,bonf
1,Visual,~,Default_mode,0.179758,0.116517,0.243000,2.530720e-08,1.012288e-07,True,bonf
2,Visual,~,Cerebellar,0.260815,0.177621,0.344009,8.012739e-10,3.205096e-09,True,bonf
3,Visual,~,Control,0.206476,0.068494,0.344457,3.357584e-03,1.343034e-02,True,bonf




AUTISMO - M28888


,lval,op,rval,Estimate,ic_inf,ic_sup,p-value,p-value-adj,rejeita,tipo_correcao
0,Visual,~,Somatomotor,0.047730,-0.054286,0.149745,3.591324e-01,1.000000e+00,False,bonf
1,Visual,~,Default_mode,0.262686,0.171756,0.353615,1.494009e-08,5.976035e-08,True,bonf
2,Visual,~,Cerebellar,0.241391,0.127461,0.355322,3.284881e-05,1.313952e-04,True,bonf
3,Visual,~,Control,0.203513,0.007589,0.399436,4.175795e-02,1.670318e-01,False,bonf


#### 10.2) P valores

In [62]:
def bootstrap_p_valores(modelo: dict, dados: pd.DataFrame, n_sims: int):
    formula = modelo["formula_sem"]
    p_valores_arestas = {a:[] for a in modelo["arestas"]}
    model = semopy.Model(formula)
    for i in tqdm(range(n_sims)):
        reamostragem = dados.sample(n=len(dados), replace=True)
        model.fit(reamostragem)
        parametros = model.inspect()
        for aresta in p_valores_arestas.keys():
            split = aresta.split("->")
            lval = split[1]
            rval = split[0]
            dados_aresta = parametros[(parametros["rval"] == rval) & (parametros["lval"] == lval)]
            p_valores_arestas[aresta].append(dados_aresta["p-value"].values[0].item())

    return p_valores_arestas

In [ ]:
p_valores_controle = bootstrap_estimativas(modelo_ganhador, df_controle, n_sims=10_000)
p_valores_autismo = bootstrap_estimativas(modelo_ganhador, df_autismo, n_sims=10_000)

100%|██████████| 10000/10000 [00:51<00:00, 194.47it/s]


#### 10.3) Testes de chow

In [77]:
def bootstrap_teste_chow(modelo: dict, dados_controle: pd.DataFrame, dados_autismo: pd.DataFrame, n_sims: int):
    formula = modelo["formula_ols"]
    lista_p_valores = []
    lista_estatisticas_f = []

    k = 5
    N_controle = len(dados_controle) 
    N_autismo = len(dados_autismo)

    for i in tqdm(range(n_sims)):
        reamostragem_controle = dados_controle.sample(n=N_controle, replace=True)
        reamostragem_autismo = dados_autismo.sample(n=N_autismo, replace=True)
        reamostragem_total = pd.concat([reamostragem_controle, reamostragem_autismo], axis=0)

        modelo_total = smf.ols(formula, data=reamostragem_total, drop_cols=retira_va_indep).fit()
        modelo_controle = smf.ols(formula, data=reamostragem_controle, drop_cols=retira_va_indep).fit()
        modelo_autismo = smf.ols(formula, data=reamostragem_autismo, drop_cols=retira_va_indep).fit()

        ssr_total = modelo_total.ssr
        ssr_controle = modelo_controle.ssr
        ssr_autismo = modelo_autismo.ssr

        numerador = (ssr_total - (ssr_controle + ssr_autismo)) / k
        denominador = (ssr_controle + ssr_autismo) / (N_controle + N_autismo - 2 * k)
        estatistica_f = numerador / denominador

        # 5. Calcular o P-valor Global:
        graus_liberdade_numerador = k # 5 variáveis
        graus_liberdade_denominador = N_controle + N_autismo - 2 * k
        p_valor = stats.f.sf(estatistica_f, graus_liberdade_numerador, graus_liberdade_denominador)

        lista_estatisticas_f.append(estatistica_f.item())
        lista_p_valores.append(p_valor.item())
    
    return lista_estatisticas_f, lista_p_valores


In [81]:
lista_f, lista_p = bootstrap_teste_chow(modelo=modelo_ganhador,
                                        dados_controle=df_controle,
                                        dados_autismo=df_autismo,
                                        n_sims=10_000)

100%|██████████| 10000/10000 [02:43<00:00, 61.32it/s]


#### 11) Teste de Permutacao

- Se o p-valor for menor que o seu nível de significância (geralmente $p < 0.05$): Você rejeita a $H_0$. Conclui-se que o modelo realmente possui qualidades de ajuste diferentes dependendo do grupo.

- Se o p-valor for maior ou igual a 0.05: Você falha em rejeitar a $H_0$. Conclui-se que não há evidências suficientes para dizer que o ajuste do modelo é diferente entre os grupos (a diferença observada pode ser apenas ruído aleatório).

In [32]:
def teste_permutacao_sem(X_A, X_B, formula, n_permutacoes=100, metrica="AIC",bicaudal=True, seed=None):
    """
    Realiza um teste de permutação para comparar o ajuste (RMSEA) de um modelo SEM 
    entre dois grupos.
    """
    if seed is not None:
        np.random.seed(seed)
        
    # ATENÇÃO: Mantemos os dados como pandas DataFrames (não convertemos para np.array)
    # pois o semopy precisa ler o nome das colunas especificados na `formula`.
    
    # --- 1. Calcula a métrica real (grupos originais) ---
    modelo_A_real = semopy.Model(formula)
    modelo_A_real.fit(X_A, obj="MLW")
    metrica_A_real = semopy.stats.calc_stats(modelo_A_real)[metrica].values[0]

    modelo_B_real = semopy.Model(formula)
    modelo_B_real.fit(X_B, obj="MLW")
    metrica_B_real = semopy.stats.calc_stats(modelo_B_real)[metrica].values[0]
    
    diff_real = metrica_A_real - metrica_B_real
    
    # --- 2. Empilha todos os dados usando pandas ---
    X_todos = pd.concat([X_A, X_B], ignore_index=True)
    
    n_A = len(X_A)
    n_total = len(X_todos)
    
    diffs_permutadas = [] # Usamos lista para permitir pular iterações que falharem
    
    # --- 3. Laço de permutações ---
    for i in range(n_permutacoes):
        # Embaralha apenas os índices das linhas
        indices = np.random.permutation(n_total)
        
        # Divide no Pseudo Grupo A e Pseudo Grupo B usando .iloc do Pandas
        X_pseudo_A = X_todos.iloc[indices[:n_A]]
        X_pseudo_B = X_todos.iloc[indices[n_A:]]
        
        try:
            modelo_pseudo_A = semopy.Model(formula)
            modelo_pseudo_A.fit(X_pseudo_A, obj="MLW")
            metrica_pseudo_A = semopy.stats.calc_stats(modelo_pseudo_A)["RMSEA"].values[0]
            
            modelo_pseudo_B = semopy.Model(formula)
            modelo_pseudo_B.fit(X_pseudo_B, obj="MLW")
            metrica_pseudo_B = semopy.stats.calc_stats(modelo_pseudo_B)["RMSEA"].values[0]
            
            # Salva a diferença
            diffs_permutadas.append(metrica_pseudo_A - metrica_pseudo_B)
            
        except Exception as e:
            print(f"Iteracao {i + 1} falhou")
            # Modelos SEM às vezes não convergem com dados embaralhados sem sentido
            # Esse bloco evita que o teste inteiro quebre por causa de uma permutação ruim
            continue
            
    # Converte de volta para array para fazer a matemática do p-valor
    diffs_permutadas = np.array(diffs_permutadas)
    
    # Se muitas falharem, avisa o usuário
    if len(diffs_permutadas) < n_permutacoes:
        print(f"Aviso: {n_permutacoes - len(diffs_permutadas)} permutações falharam ao convergir e foram ignoradas.")
        
    # --- 4. Calcula o p-valor ---
    if bicaudal:
        p_valor = np.mean(np.abs(diffs_permutadas) >= np.abs(diff_real))
    else:
        p_valor = np.mean(diffs_permutadas >= diff_real)
        
    return p_valor, diff_real, diffs_permutadas

In [34]:
modelos = ["M25888_Edges4", "M10302_Edges4", "M03528_Edges4", "M00934_Edges4", "M00001_Edges4"] 
for modelo in modelos:
    formula = formula_modelos_conexos[modelo]["formula"]
    p_valor, diff_real, diffs_permutadas = teste_permutacao_sem(df_controle, df_autismo, formula, metrica="BIC", n_permutacoes=1000, bicaudal=True)
    print(f"\n\nModelo: {modelo}")
    print(f"P valor: {p_valor}")
    print(f"Diferenca real: {diff_real}")



Modelo: M25888_Edges4
P valor: 0.0
Diferenca real: 5.566837109350331


Modelo: M10302_Edges4
P valor: 0.0
Diferenca real: 5.565866071515124


Modelo: M03528_Edges4
P valor: 0.0
Diferenca real: 5.566485650774332


Modelo: M00934_Edges4
P valor: 0.0
Diferenca real: 5.566493240450519


Modelo: M00001_Edges4
P valor: 0.0
Diferenca real: 5.566493411142005
